<a href="https://colab.research.google.com/github/dxda6216/q10_two_temperatures/blob/main/circadian_period_q10_2_temperatures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Circadian period Q10 calculator (Colab version)

<a href="https://colab.research.google.com/github/dxda6216/q10_two_temperatures/blob/main/circadian_period_q10_2_temperatures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Calculates the temperature coefficient (Q10) of the circadian period from measurements at two temperatures. Replicate measurements at each temperature are supported.

**Prefer not to run code?** The same calculator is available as a web app: <https://dxda6216.github.io/q10_two_temperatures/index.html>

Q10 is calculated from the mean periods τ̄₁ and τ̄₂ at temperatures T₁ and T₂:

$$Q_{10} = \left(\frac{\bar\tau_1}{\bar\tau_2}\right)^{\frac{10}{T_2 - T_1}}$$

### How to use

1. Run **Step 1** once to load the calculator.
2. In **Step 2**, replace the sample data with your own values in the form on the right, then run the cell.
3. (Optional) Run **Step 3** to download the results as files.

Your data stays in your own Colab session; nothing is sent elsewhere by this notebook.

In [ ]:
#@title Step 1: Load the calculator (run once)
#@markdown Defines the calculation, plotting and export functions. No input needed.

import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COOL, WARM, INK = "#2F6FA3", "#B8691F", "#1D2A33"
LOW, HIGH = 0.9, 1.1  # typical temperature-compensated range

class InputError(ValueError):
    pass

def parse_periods(text):
    """Split on spaces, commas, semicolons, tabs or line breaks."""
    tokens = [t for t in re.split(r"[\s,;、，]+", str(text)) if t]
    values, bad = [], []
    for t in tokens:
        try:
            v = float(t.replace("．", "."))
            if math.isfinite(v):
                values.append(v)
            else:
                bad.append(t)
        except ValueError:
            bad.append(t)
    return values, bad

def summarize(values):
    a = np.asarray(values, dtype=float)
    n = a.size
    mean = a.mean()
    sd = a.std(ddof=1) if n > 1 else float("nan")
    sem = sd / math.sqrt(n) if n > 1 else float("nan")
    return {"n": n, "mean": mean, "sd": sd, "sem": sem}

def calculate(T1, periods1, T2, periods2):
    """Validate inputs and return a dict of all results."""
    p1, bad1 = parse_periods(periods1)
    p2, bad2 = parse_periods(periods2)
    if bad1 or bad2:
        raise InputError("Not a number: " + ", ".join((bad1 + bad2)[:3]))
    if not p1 or not p2:
        raise InputError("Enter at least one period for each temperature.")
    if T1 == T2:
        raise InputError("The two temperatures must differ.")
    if min(p1 + p2) <= 0:
        raise InputError("Periods must be greater than zero.")

    s1, s2 = summarize(p1), summarize(p2)
    dT = T2 - T1
    q10 = (s1["mean"] / s2["mean"]) ** (10 / dT)

    # Approximate SE via the delta method on ln Q10 (needs n >= 2 at both temperatures)
    se = lo = hi = None
    if s1["n"] > 1 and s2["n"] > 1:
        se_ln = (10 / abs(dT)) * math.sqrt((s1["sem"] / s1["mean"]) ** 2 + (s2["sem"] / s2["mean"]) ** 2)
        se = q10 * se_ln
        lo, hi = q10 * math.exp(-1.96 * se_ln), q10 * math.exp(1.96 * se_ln)

    if LOW <= q10 <= HIGH:
        label = f"Within the typical compensated range ({LOW}–{HIGH})."
    elif q10 > HIGH:
        label = f"Above the typical compensated range ({LOW}–{HIGH})."
    else:
        label = f"Below {LOW}: overcompensated."

    # Fitted curve through both means: tau(T) = tau1 * Q10^(-(T - T1)/10)
    t_lo, t_hi = min(T1, T2), max(T1, T2)
    pad = (t_hi - t_lo) * 0.25
    t_fit = np.linspace(t_lo - pad, t_hi + pad, 101)
    p_fit = s1["mean"] * q10 ** (-(t_fit - T1) / 10)

    return dict(T1=T1, T2=T2, p1=p1, p2=p2, s1=s1, s2=s2, q10=q10, se=se, lo=lo, hi=hi,
                label=label, t_fit=t_fit, p_fit=p_fit)

def tables(r):
    replicates = pd.DataFrame(
        [(r["T1"], v) for v in r["p1"]] + [(r["T2"], v) for v in r["p2"]],
        columns=["Temperature (°C)", "Period (h)"])
    summary = pd.DataFrame(
        [(r["T1"], r["s1"]["n"], r["s1"]["mean"], r["s1"]["sd"], r["s1"]["sem"]),
         (r["T2"], r["s2"]["n"], r["s2"]["mean"], r["s2"]["sd"], r["s2"]["sem"])],
        columns=["Temperature (°C)", "n", "Mean period (h)", "SD", "SEM"])
    fitted = pd.DataFrame({"Temperature (°C)": r["t_fit"], "Fitted period (h)": r["p_fit"]})
    return replicates, summary, fitted

def tsv(df, decimals, temp_decimals=2):
    """Tab-delimited text; temperatures to 2 decimals, other numbers to `decimals`."""
    out = df.copy()
    for c in out.columns:
        if c == "n":
            continue
        d = temp_decimals if c.startswith("Temperature") else decimals
        out[c] = out[c].map(lambda v: "" if pd.isna(v) else f"{v:.{d}f}")
    return out.to_csv(sep="\t", index=False)

def summary_tsv(summary, q10):
    return tsv(summary, 4) + f"\nQ10\t{q10:.4f}\n"

def plot(r, title=""):
    fig, ax = plt.subplots(figsize=(7, 4.6), dpi=110)
    ax.plot(r["t_fit"], r["p_fit"], color=INK, lw=2, label="Fitted curve", zorder=1)
    span = r["t_fit"][-1] - r["t_fit"][0]
    off = span * 0.03  # replicates sit just left of the mean so they don't hide it
    for T, vals, s, col in ((r["T1"], r["p1"], r["s1"], COOL), (r["T2"], r["p2"], r["s2"], WARM)):
        jitter = [((i % 3) - 1) * off * 0.2 for i in range(len(vals))] if len(vals) > 1 else [0]
        ax.scatter([T - off + j for j in jitter], vals, s=30, facecolors="none", edgecolors=col, lw=1.5, zorder=2)
        if s["n"] > 1:
            ax.errorbar(T, s["mean"], yerr=s["sd"], fmt="none", ecolor=col, elinewidth=1.5, capsize=5, zorder=3)
        ax.scatter([T], [s["mean"]], s=80, color=col, edgecolors="white", lw=1.5, zorder=4)
    # legend proxies in neutral grey
    ax.scatter([], [], s=30, facecolors="none", edgecolors="#5C6B76", label="Replicate")
    ax.scatter([], [], s=80, color="#5C6B76", label="Mean ± SD")
    ax.text(0.98, 0.95, f"Q10 = {r['q10']:.3f}", transform=ax.transAxes, ha="right", va="top", fontsize=11, color=INK)
    ax.set_xlim(r["t_fit"][0], r["t_fit"][-1])
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Period (h)")
    if title:
        ax.set_title(title, loc="left", fontsize=12)
    ax.grid(axis="y", color="#CCD5DB", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(loc="lower left", frameon=False, fontsize=9)
    fig.tight_layout()
    return fig

def report(r, description=""):
    s1, s2 = r["s1"], r["s2"]
    def line(T, s):
        txt = f"  {T:g} °C: n = {s['n']}, mean {s['mean']:.3f} h"
        return txt + (f", SD {s['sd']:.3f}, SEM {s['sem']:.3f}" if s["n"] > 1 else "")
    print(description or "Results")
    print(line(r["T1"], s1)); print(line(r["T2"], s2)); print()
    if r["se"] is not None:
        print(f"Q10 (from mean periods) = {r['q10']:.3f} ± {r['se']:.3f}")
        print(f"  ± is the approximate SE from SEMs; 95% range {r['lo']:.3f}–{r['hi']:.3f}")
    else:
        print(f"Q10 (from mean periods) = {r['q10']:.3f}")
        if s1["n"] > 1 or s2["n"] > 1:
            print("  Add replicates at both temperatures for an SE.")
    print("  " + r["label"])

print("Calculator loaded. Now go to Step 2.")

In [ ]:
#@title Step 2: Enter your data and calculate
#@markdown **The values below are sample data.** Replace them with your own, then run this cell.
#@markdown
#@markdown Separate replicate periods with spaces, commas or semicolons (e.g. `27.2, 27.5, 26.9`). Use a decimal point, not a decimal comma. A single value per temperature is fine.

description = "" #@param {type:"string"}
#@markdown ---
#@markdown **Condition 1**
temperature_1 = 33.4 #@param {type:"number"}
periods_1 = "27.2, 27.5, 26.9" #@param {type:"string"}
#@markdown **Condition 2**
temperature_2 = 37.0 #@param {type:"number"}
periods_2 = "22.6, 22.9, 22.4" #@param {type:"string"}
#@markdown ---
show_tab_delimited = True #@param {type:"boolean"}

try:
    result = calculate(float(temperature_1), periods_1, float(temperature_2), periods_2)
except InputError as e:
    result = None
    print("⚠ " + str(e))

if result:
    report(result, description)
    fig = plot(result, description)
    plt.show()
    replicates, summary, fitted = tables(result)
    if show_tab_delimited:
        print("Tab-delimited data: select a block, copy, and paste into Excel (Paste Special → Unicode Text).\n")
        print("=== All replicates ===")
        print(tsv(replicates, 3))
        print("=== Summary ===")
        print(summary_tsv(summary, result["q10"]))
        print("=== Fitted curve ===")
        print(tsv(fitted, 4, 3))

In [ ]:
#@title Step 3 (optional): Download results as files
#@markdown Saves the results from Step 2. Tick what you want, then run this cell.
download_tables = True #@param {type:"boolean"}
download_plot_png = True #@param {type:"boolean"}
download_plot_svg = False #@param {type:"boolean"}
file_prefix = "q10" #@param {type:"string"}

if "result" not in globals() or result is None:
    print("Run Step 2 successfully first.")
else:
    prefix = re.sub(r"[^\w\-]+", "_", file_prefix.strip()) or "q10"
    files_out = []
    if download_tables:
        open(f"{prefix}_replicates.tsv", "w", encoding="utf-8").write(tsv(replicates, 3))
        with open(f"{prefix}_summary.tsv", "w", encoding="utf-8") as fh:
            fh.write(summary_tsv(summary, result["q10"]))
        open(f"{prefix}_fitted_curve.tsv", "w", encoding="utf-8").write(tsv(fitted, 4, 3))
        files_out += [f"{prefix}_replicates.tsv", f"{prefix}_summary.tsv", f"{prefix}_fitted_curve.tsv"]
    if download_plot_png:
        fig.savefig(f"{prefix}_plot.png", dpi=300, bbox_inches="tight"); files_out.append(f"{prefix}_plot.png")
    if download_plot_svg:
        fig.savefig(f"{prefix}_plot.svg", bbox_inches="tight"); files_out.append(f"{prefix}_plot.svg")
    try:
        from google.colab import files
        for f in files_out:
            files.download(f)
    except ImportError:
        print("Not running in Colab; files saved to the working directory:")
    for f in files_out:
        print("  " + f)

## Calculation details

- **Fitted curve:** τ(T) = τ̄₁ · Q10^(−(T − T₁)/10), passing through both mean periods; drawn from 25% of the temperature span below the lower temperature to 25% above the higher one (101 points).
- **Summary statistics:** n, mean, sample SD (n − 1 denominator), SEM = SD / √n.
- **Uncertainty of Q10** (when both temperatures have n ≥ 2), delta method on ln Q10:
  SE(ln Q10) = (10 / |T₂ − T₁|) · √[(SEM₁/τ̄₁)² + (SEM₂/τ̄₂)²]; SE(Q10) ≈ Q10 · SE(ln Q10); 95% range ≈ Q10 · exp(±1.96 · SE(ln Q10)). This is a large-sample approximation; with small n consider a t-based interval or bootstrap.
- **Interpretation:** 0.9–1.1 within the typical compensated range; > 1.1 above it; < 0.9 overcompensated.

For data at more than two temperatures, see [dxda6216/q10](https://github.com/dxda6216/q10).

License: GPL-3.0